<h1>Extracting Data from Flight Labs API</h1>

In [1]:
import os
from dotenv import load_dotenv
from utils import extract, transform, load_to_csv, load_to_postgres
from sqlalchemy import create_engine

load_dotenv()
access_key = os.getenv('ACCESS_KEY')

# Define API endpoint
url = 'https://www.goflightlabs.com/flights'

# Extracting data from API endpoint
flight_data_raw = extract(url, access_key)
print(flight_data_raw.shape)

Returned status code: 200
Extraction complete
(100, 23)


<H1>Exploring the Raw Data </h1>

In [2]:
# EDA
display(flight_data_raw.head(), flight_data_raw.info(), flight_data_raw.describe())

print('\nNumber of null values for every column feature\n')
flight_data_raw.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   hex            100 non-null    object 
 1   reg_number     99 non-null     object 
 2   flag           100 non-null    object 
 3   lat            100 non-null    float64
 4   lng            100 non-null    float64
 5   alt            100 non-null    int64  
 6   dir            100 non-null    float64
 7   speed          100 non-null    int64  
 8   v_speed        100 non-null    int64  
 9   flight_number  100 non-null    object 
 10  flight_icao    100 non-null    object 
 11  flight_iata    100 non-null    object 
 12  dep_icao       100 non-null    object 
 13  dep_iata       100 non-null    object 
 14  arr_icao       100 non-null    object 
 15  arr_iata       100 non-null    object 
 16  airline_icao   100 non-null    object 
 17  airline_iata   100 non-null    object 
 18  aircraft_ic

,hex,reg_number,flag,lat,lng,alt,dir,speed,v_speed,flight_number,...,dep_iata,arr_icao,arr_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type,squawk
0,A6981A,N524DE,US,45.914328,-119.303820,11198,311.1,704,0,438,...,FLL,KSEA,SEA,DAL,DL,A21N,1764876753,en-route,adsb,NaN
1,789280,B-KKD,HK,25.082999,121.400468,10345,230.5,747,0,615,...,ICN,VHHH,HKG,HKE,UO,A21N,1764876752,en-route,adsb,NaN
2,A24508,N24505,US,26.629817,-81.652149,7441,132.1,774,0,1775,...,ORD,KFLL,FLL,UAL,UA,A21N,1764876753,en-route,adsb,NaN
3,39348E,F-GNEO,FR,45.274424,12.918490,10383,283.0,813,0,3069,...,SKG,LFPO,ORY,TVF,TO,A20N,1764876753,en-route,adsb,NaN
4,346391,EC-NFA,ES,28.067591,-15.312638,1594,41.8,477,0,6014,...,LPA,LEMD,MAD,IBB,NT,E295,1764876753,en-route,adsb,NaN


None

,lat,lng,alt,dir,speed,v_speed,updated
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.0,1.000000e+02
mean,19.074086,12.164671,9483.310000,163.172000,761.800000,0.0,1.764877e+09
std,26.813601,71.874257,3370.238066,104.299266,175.775798,0.0,5.024184e-01
min,-41.321808,-163.209594,18.000000,2.000000,24.000000,0.0,1.764877e+09
25%,-0.742606,-45.408428,8562.000000,74.125000,706.250000,0.0,1.764877e+09
50%,24.372844,18.960705,10931.500000,148.250000,793.000000,0.0,1.764877e+09
75%,38.875391,72.399117,11681.750000,250.475000,875.000000,0.0,1.764877e+09
max,64.813354,174.807268,13012.000000,358.900000,1038.000000,0.0,1.764877e+09



Number of null values for every column feature



hex               0
reg_number        1
flag              0
lat               0
lng               0
alt               0
dir               0
speed             0
v_speed           0
flight_number     0
flight_icao       0
flight_iata       0
dep_icao          0
dep_iata          0
arr_icao          0
arr_iata          0
airline_icao      0
airline_iata      0
aircraft_icao     0
updated           0
status            0
type              0
squawk           99
dtype: int64

<h1>Data Cleaning</h1>

<li>Replacing missing values in "squawk" column with "unknown" if the column is pulled during extraction</li>
<li>Replacing missing values in 'alt', 'speed' and v_speed to 0</li>
<li>Renaming columns</li>
<li>Converting "updated" values to datetime</li>


In [3]:
flight_data_clean = transform(flight_data_raw)

display(flight_data_clean.head())

print('\nNumber of null values for every column feature\n')
flight_data_clean.isnull().sum()


transform complete


,hex,reg_number,flag,latitude,longitude,altitude_ft,dir,speed_mph,v_speed_mph,flight_number,...,departure_iata,arrival_icao,arrival_iata,airline_icao,airline_iata,aircraft_icao,updated,status,type,squawk
0,A6981A,N524DE,US,45.914328,-119.303820,36738.84632,311.1,437.445184,0.0,438,...,FLL,KSEA,SEA,DAL,DL,A21N,2025-12-04 14:32:33,en-route,adsb,Unknown
1,789280,B-KKD,HK,25.082999,121.400468,33940.28980,230.5,464.164137,0.0,615,...,ICN,VHHH,HKG,HKE,UO,A21N,2025-12-04 14:32:32,en-route,adsb,Unknown
2,A24508,N24505,US,26.629817,-81.652149,24412.73044,132.1,480.941154,0.0,1775,...,ORD,KFLL,FLL,UAL,UA,A21N,2025-12-04 14:32:33,en-route,adsb,Unknown
3,39348E,F-GNEO,FR,45.274424,12.918490,34064.96172,283.0,505.174623,0.0,3069,...,SKG,LFPO,ORY,TVF,TO,A20N,2025-12-04 14:32:33,en-route,adsb,Unknown
4,346391,EC-NFA,ES,28.067591,-15.312638,5229.65896,41.8,296.393967,0.0,6014,...,LPA,LEMD,MAD,IBB,NT,E295,2025-12-04 14:32:33,en-route,adsb,Unknown



Number of null values for every column feature



hex               0
reg_number        1
flag              0
latitude          0
longitude         0
altitude_ft       0
dir               0
speed_mph         0
v_speed_mph       0
flight_number     0
flight_icao       0
flight_iata       0
departure_icao    0
departure_iata    0
arrival_icao      0
arrival_iata      0
airline_icao      0
airline_iata      0
aircraft_icao     0
updated           0
status            0
type              0
squawk            0
dtype: int64

<h1>Loading Cleaned Data to CSVs and Postgres </h1>

In [5]:
# Connecting to local flight data database
dbname=os.getenv('DB_NAME')
user=os.getenv('DB_USER')
password=os.getenv('DB_PASSWORD')
host='localhost'
port=os.getenv('DB_PORT')

conn = create_engine(f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}')

# Loading raw and clean data
load_to_csv(flight_data_raw, flight_data_clean)
load_to_postgres(flight_data_raw, flight_data_clean, conn)

load complete
